# [1장 2강 심화] - 딥러닝 적용 판단 실습

## 실습 배경

- 루멘 LLM 플랫폼 팀은 사내 문서 질의응답 서비스에 들어오는 문서를 `업무 규정`, `복지`, `보안`, `기타`로 분류하려 한다.
- 기획팀은 "딥러닝이면 정확도가 높다"며, 네 종류 전부를 딥러닝 모델 하나로 분류하자고 제안했다.
- 그러나 분류 대상의 성격은 제각각이다.
- 어떤 문서는 규칙이 명확하다. 제목이 `[보안등급:대외비]`로 시작하면 예외 없이 보안 문서이므로, 단순 규칙만으로 충분하다.
- 반면 어떤 문서는 표현이 제각각이다. 같은 의도라도 `계정이 잠겼다`, `로그인이 막혔다`, `인증에 계속 실패한다`처럼 다양하게 적히는데, 단어만 세는 규칙은 이들을 서로 다른 사례로 본다.
- 따라서 팀은 문제마다 규칙 기반, 사람이 특징을 설계하는 머신러닝, 표현까지 학습하는 딥러닝 중 무엇을 먼저 검토할지 가려내야 한다.
- 이번 실습의 목표는 모델 이름을 고르는 것이 아니다.
- 데이터 형태, 규칙 안정성, 라벨 수, 오판 비용을 근거로 후보를 좁히고, 딥러닝을 쓰더라도 검증 데이터에서 단순 기준선과 반드시 비교한다는 원칙을 세운다.

## 실습 목표

- 규칙 기반·머신러닝·딥러닝 접근이 각각 어떤 상황에 맞는지 사례별로 구분한다.
- 원본 텍스트의 표현이 다양할 때, 사람이 직접 만든 특징이 어떤 한계를 갖는지 설명한다.
- 데이터와 운영상의 제약을 근거로 적용 여부 판단 메모를 작성한다.
- "딥러닝을 쓸 수 있다"와 "딥러닝을 먼저 써야 한다"를 구분한다.

## 진행 방식

- 각 문제는 독립적으로 푼다. 앞 문제의 출력이 다음 문제의 입력이 되지 않는다.
- 주어진 수치는 모두 같은 검증 묶음에서 얻은 것으로 가정한다.
- 아직 배우지 않은 모델 내부 구조는 판단 근거로 쓰지 않는다.

## 상황 자료

```
업무                 데이터          현재 규칙             라벨 데이터
보안등급 라우팅      제목 고정 접두어  변경이 드물고 명확함       400건
상담 주제 분류       자유 형식 문장    키워드 38개             28,000건
신규 제품 불만 탐지  짧은 후기         규칙 없음                 120건
```

### 문제 1. 키워드 규칙의 조용한 실패 찾기
#### 업무 요청
- 상담 주제 분류 규칙은 에러 없이 실행됩니다.
- 하지만 문맥을 고려하지 못해 엉뚱한 곳으로 라우팅하는 경우가 있습니다.
- 코드와 검증 로그를 살펴보고, 어떤 유형의 실패가 일어나는지 특정한 뒤 최소 두 가지 사례를 설명하세요.

### 수행해야 할 작업

1. 각 문장을 현재 규칙으로 분류한다.
2. 분류 결과가 정답과 다른 사례의 인덱스를 찾는다.
3. 단어의 등장 여부만 보는 규칙이 어떤 문맥을 놓쳤는지 설명한다.
4. 이 규칙을 곧바로 폐기할지, 아니면 딥러닝 후보와 비교한 뒤 판단할지 결정한다.

In [20]:
texts = ["환불이 아니라 교환을 원합니다", "로그인이 막혀 결제가 안 됩니다", "환불 부탁드립니다"]
labels = ["교환", "계정", "환불"]

def current_rule(text):
    if "환불" in text:
        return "환불"
    if "결제" in text:
        return "결제"
    return "기타"

preds = [current_rule(text) for text in texts]
preds

# zip()과 enumerate()는 결과를 바로 리스트로 만들어주는 함수가 아니라 iterator(반복 가능한 객체) 를 반환
wrong = [i for i, (pred, label) in enumerate(zip(preds, labels)) if pred != label]

# wrong = []
# for i in range(len(preds)):
#     pred = preds[i]
#     label = labels[i]
#     if pred != label:
#         wrong.append(i)

print(f"분류 결과: {preds}")
print(f"다른 사례: {wrong}")



분류 결과: ['환불', '결제', '환불']
다른 사례: [0, 1]


- "환불이 아니라 교환을 원합니다" -> 실제 교환을 원하지만 "환불"이라는 단어만 보고 환불로 잘 못 분류 됨
- "로그인이 막혀 결제가 안 됩니다" -> "로그인" 단어를 분류하는게 없고, "결제" 단어로 잘 못 분류 됨
- 이 진단은 키워드 규칙의 문맥 손실을 보여줄 뿐 딥러닝 모델의 우위를 증명하지는 못한다.
- 실제 교체 판단에는 더 다양한 라벨 데이터와 검증 환경을 고정한 비교 실험이 추가로 필요하다.

## 문제 2. 적용 판단 도우미 작성하기
- 팀은 새로운 분류 문제를 만날 때마다 "어떤 접근부터 시도할지"를 반복해서 논의한다. 이 과정에서 데이터 양, 규칙 안정성, 오판 비용 같은 핵심 조건을 매번 일관되게 점검하지 못하는 문제가 있다.
- 팀 리드는 이런 누락을 줄이기 위해, 판단에 필요한 조건을 자동으로 짚어주는 1차 판단 도우미를 요청했다.
- 이 함수는 최종 결정권자가 아니라, 어떤 접근부터 검증할지 알려주는 체크리스트 역할이다.

### 수행해야 할 작업

1. 규칙이 안정적이고 예외가 거의 없으면 `rule_first`를 반환한다.
2. 자유 형식의 원본 텍스트이고 라벨이 5,000건 이상이면 'deep_learning_candidate'를 반환한다.
3. 두 경우 모두 아니면 'simple_baseline_first'를 반환한다.
4. 세 가지 사례를 모두 실행해보고, 각각 왜 그 분기로 갈라졌는지 이유를 설명한다.

In [ ]:
# ## 상황 자료

# ```
# 업무                 데이터          현재 규칙             라벨 데이터
# 보안등급 라우팅      제목 고정 접두어  변경이 드물고 명확함       400건
# 상담 주제 분류       자유 형식 문장    키워드 38개             28,000건
# 신규 제품 불만 탐지  짧은 후기         규칙 없음                 120건
# ```

def recommend_start(raw_unstructured, labeled_count, stable_rule):
    if stable_rule:
        return "rule_first"
    
    if raw_unstructured and labeled_count >= 5000:
        return "deep_learning_candidate"
    
    return "simple_baseline_first"

cases = [
    (False, 400, True),
    (True, 20000, False),
    (True, 120, False)
    ]

# * unpacking operator
for case in cases:
    print(recommend_start(*case))

rule_first
deep_learning_candidate
simple_baseline_first


- '보안등급 라우팅' 업무는 규칙이 명확하고 변경이 드물기에 규칙기반으로 고려한다.
- '상담 주제 분류' 업무는 자유 형식 문장으로 데이터가 고정이 안 되어 있고, 데이터가 5000건 이상임으로 딥러닝 모델을 후보로 올린다.
- '신규 제품 불만 탐지' 업무는 규칙이 없고 자유 형식의 텍스트이나 데이터가 120건으로 간단한 베이스 모델로 시작할 것을 고려한다.

### 문제 3. 불완전한 검증 근거로 승인과 재측정 구분하기
- 상담 주제 분류를 두고, 동일한 검증 데이터 2,000건에서 세 후보를 비교했다.
- 운영팀의 승인 기준은 정확도 0.90 이상, 문서당 처리 시간 12ms 이하다.
- 단, 처리 시간(latency)은 같은 조건에서 최소 3회 반복 측정해야만 승인 자료로 인정한다.
- 각 후보를 수치 미달인지 증거 부족인지 구분해, 승인·재측정·제외로 분류하라.

### 수행해야 할 작업
1. 각 후보가 정확도 기준과 처리 시간(latency) 기준을 각각 만족하는지 검사한다.
2. 수치는 기준을 통과했지만 반복 측정이 부족한 후보는 재측정으로 따로 분류한다.
3. 승인 기준을 만족하는 후보가 없으면, 그나마 나은 안을 억지로 승인하지 않는다.
4. 각 후보에 대한 다음 조치와, 이번 검증으로 판단할 수 있는 범위의 한계를 함께 보고한다.

### 상황 로그

```
후보  접근                  검증 정확도  처리 시간  latency 반복
A     수작업 특징 기준선       0.884       3ms         3회
B     딥러닝 후보             0.921      10ms         1회
C     작은 딥러닝 후보         0.908      13ms         3회
```

In [27]:
candidates = {
    "A": {"accuracy": 0.884, "latency_ms": 3, "latency_runs": 3},
    "B": {"accuracy": 0.921, "latency_ms": 10, "latency_runs": 1},
    "C": {"accuracy": 0.908, "latency_ms": 13, "latency_runs": 3},
}

# - 운영팀의 승인 기준은 정확도 0.90 이상, 문서당 처리 시간 12ms 이하다.
# - 단, 처리 시간(latency)은 같은 조건에서 최소 3회 반복 측정해야만 승인 자료로 인정한다.
# - 각 후보를 수치 미달인지 증거 부족인지 구분해, 승인·재측정·제외로 분류하라.

results = {}
for name, spec in candidates.items():
    if spec["accuracy"] < 0.9:
        results[name] = "제외 (정확도 미달)"
    elif spec["latency_ms"] > 12:
        results[name] = "제외 (처리 시간 초과)"
    elif spec["latency_runs"] < 3:
        results[name] = "재측정 (반복 측정 부족)"
    else:
        results[name] = "승인"

print(results)

{'A': '제외 (정확도 미달)', 'B': '재측정 (반복 측정 부족)', 'C': '제외 (처리 시간 초과)'}


- 현재 결정은 보류이며 다음 조치는 B를 같은 조건에서 두 번 더 측정하는 것이다.